# Effective Rank (Roy & Vetterli, 2007)

Implementación del *effective rank*, referenciado en el capítulo de ViT (bloque A.1: similitud + rango efectivo entre cabezas de atención).

Dada una matriz $A$ con valores singulares $\sigma_1 \geq \sigma_2 \geq \dots \geq \sigma_Q \geq 0$ ($Q = \min(M, N)$):

$$p_k = \frac{\sigma_k}{\|\sigma\|_1}, \qquad H(p_1,\dots,p_Q) = -\sum_k p_k \log p_k, \qquad \mathrm{erank}(A) = \exp\{H(p_1,\dots,p_Q)\}$$

con la convención $0 \log 0 = 0$. Cumple $1 \leq \mathrm{erank}(A) \leq \mathrm{rank}(A) \leq Q$.

In [1]:
import math

import torch


def effective_rank(A: torch.Tensor) -> torch.Tensor:
    """
    Effective rank de Roy & Vetterli (2007): erank(A) = exp(H(p)),
    donde p_k = sigma_k / ||sigma||_1 son los valores singulares de A
    normalizados y H es su entropía de Shannon (con la convención 0*log(0) = 0).

    Soporta batches: A puede tener shape (..., M, N), y el rango efectivo
    se calcula sobre las últimas dos dimensiones, devolviendo shape (...,).
    """
    singular_values = torch.linalg.svdvals(A)  # (..., Q)
    p = singular_values / singular_values.sum(dim=-1, keepdim=True)

    log_p = torch.where(p > 0, torch.log(p), torch.zeros_like(p))
    entropy = -(p * log_p).sum(dim=-1)

    return torch.exp(entropy)

## Pruebas con dos matrices de referencia

1. **Identidad $4\times4$**: los 4 valores singulares son iguales (=1), la distribución $p_k$ es uniforme y $H(p) = \log 4$, por lo que $\mathrm{erank}(I_4) = 4$.
2. **Matriz circulante Hermitiana** del ejemplo del propio paper (Sección 2.3), con valores singulares $(1+|\rho|)^2, 1-|\rho|^2, 1-|\rho|^2, (1-|\rho|)^2$, cuya forma cerrada es $\mathrm{erank}(A) = \exp\{2H(\frac{1+|\rho|}{2}, \frac{1-|\rho|}{2})\}$.

In [2]:
def binary_entropy(p: float) -> float:
    q = 1 - p
    terms = [x * math.log(x) for x in (p, q) if x > 0]
    return -sum(terms)



In [3]:
n = 4
A = torch.eye(n)
erank = effective_rank(A).item()
expected = float(n)
print(f"[identidad {n}x{n}]     erank = {erank:.6f}  (esperado {expected:.6f})")
assert math.isclose(erank, expected, rel_tol=1e-5)

[identidad 4x4]     erank = 4.000000  (esperado 4.000000)


In [4]:
rho = 0.5
A = torch.tensor([
    [1, rho, rho**2, rho],
    [rho, 1, rho, rho**2],
    [rho**2, rho, 1, rho],
    [rho, rho**2, rho, 1],
])

erank = effective_rank(A).item()
expected = math.exp(2 * binary_entropy((1 + abs(rho)) / 2))
print(f"[circulante rho={rho}] erank = {erank:.6f}  (esperado {expected:.6f})")
assert math.isclose(erank, expected, rel_tol=1e-5)


[circulante rho=0.5] erank = 3.079201  (esperado 3.079201)


## Rango efectivo de las cabezas de atención de un modelo entrenado

Modelo ViT entrenado en MNIST (`model.pt`, solo pesos). El checkpoint no trae `config.json`, así que la arquitectura se infiere de las shapes de los tensores guardados: `hidden_size=48`, `num_hidden_layers=4`, `num_attention_heads=4` (head_size=12), `intermediate_size=192`, `patch_size=4`, `image_size=32`, `num_channels=3`, `num_classes=10`, sin `faster_attention` (cabezas separadas con `.query`/`.key`/`.value`).

In [5]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parents[1]
print(f"Project root: {project_root}")
sys.path.append(str(project_root))

import torch

from modules.model import ViTForClassfication

Project root: /Users/valefeve/Projects/interpretability of neural networks/interpretable-transformers


In [6]:
config = {
    "hidden_size": 48,
    "num_hidden_layers": 4,
    "num_attention_heads": 4,
    "intermediate_size": 192,
    "hidden_dropout_prob": 0.0,
    "attention_probs_dropout_prob": 0.0,
    "initializer_range": 0.02,
    "image_size": 32,
    "patch_size": 4,
    "num_channels": 3,
    "num_classes": 10,
    "qkv_bias": True,
    "use_faster_attention": False,
}

model = ViTForClassfication(config)
state_dict = torch.load("model.pt", map_location="cpu")
model.load_state_dict(state_dict)
model.eval()

ViTForClassfication(
  (embedding): Embeddings(
    (patch_embeddings): PatchEmbeddings(
      (projection): Conv2d(3, 48, kernel_size=(4, 4), stride=(4, 4))
    )
    (dropout): Dropout(p=0.0, inplace=False)
  )
  (encoder): Encoder(
    (blocks): ModuleList(
      (0-3): 4 x Block(
        (attention): MultiHeadAttention(
          (heads): ModuleList(
            (0-3): 4 x AttentionHead(
              (query): Linear(in_features=48, out_features=12, bias=True)
              (key): Linear(in_features=48, out_features=12, bias=True)
              (value): Linear(in_features=48, out_features=12, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (output_projection): Linear(in_features=48, out_features=48, bias=True)
          (output_dropout): Dropout(p=0.0, inplace=False)
        )
        (layernorm_1): LayerNorm((48,), eps=1e-05, elementwise_affine=True, bias=True)
        (mlp): MLP(
          (dense_1): Linear(in_features=48, out

In [7]:
last_block = model.encoder.blocks[-1]
heads = last_block.attention.heads

query_weights = torch.stack([head.query.weight.detach() for head in heads])  # (num_heads, head_size, hidden_size)
key_weights = torch.stack([head.key.weight.detach() for head in heads])
value_weights = torch.stack([head.value.weight.detach() for head in heads])

print(f"{len(heads)} cabezas de atención en el último bloque, cada matriz con shape {tuple(query_weights.shape[1:])}")
print("\nQuery weights por cabeza:")
print(query_weights)
print("\nKey weights por cabeza:")
print(key_weights)
print("\nValue weights por cabeza:")
print(value_weights)

4 cabezas de atención en el último bloque, cada matriz con shape (12, 48)

Query weights por cabeza:
tensor([[[-0.2291, -0.1780,  0.3583,  ...,  0.2037, -0.2656,  0.1339],
         [-0.1131,  0.0604, -0.2410,  ...,  0.0264, -0.5287,  0.0772],
         [ 0.5420, -0.3695, -0.3649,  ..., -0.1642,  0.4750,  0.4939],
         ...,
         [-0.3528,  0.2980,  0.2032,  ...,  0.1384, -0.2366,  0.0783],
         [ 0.1911, -0.3010,  0.0577,  ...,  0.2775,  0.0901,  0.0967],
         [ 0.1895,  0.0067, -0.1698,  ...,  0.1204, -0.0930, -0.1797]],

        [[-0.8964,  0.1974, -0.0554,  ..., -0.5615,  0.6035, -0.4337],
         [-0.6277,  0.0586, -0.0784,  ..., -0.0959,  0.3321, -0.5254],
         [ 0.1581,  0.2315,  0.1012,  ..., -0.0458, -0.1851,  0.1605],
         ...,
         [-0.0437,  0.3215, -0.3140,  ..., -0.1167, -0.5517,  0.4733],
         [-0.0539,  0.2547, -0.0547,  ...,  0.1062, -0.3981,  0.3137],
         [-0.5351, -0.1965,  0.1460,  ...,  0.9475,  0.7021, -0.0982]],

        [[ 0.07

In [12]:
erank_query = effective_rank(query_weights)
erank_key = effective_rank(key_weights)
erank_value = effective_rank(value_weights)

for h in range(len(heads)):
    print(f"head {h}: erank(Q)={erank_query[h]:.4f}  erank(K)={erank_key[h]:.4f}  erank(V)={erank_value[h]:.4f}")

head 0: erank(Q)=8.8387  erank(K)=7.5238  erank(V)=10.5189
head 1: erank(Q)=8.5091  erank(K)=7.7325  erank(V)=10.3649
head 2: erank(Q)=9.4800  erank(K)=8.6640  erank(V)=10.3762
head 3: erank(Q)=7.4286  erank(K)=6.2817  erank(V)=10.6534


In [10]:
model.encoder.blocks

ModuleList(
  (0-3): 4 x Block(
    (attention): MultiHeadAttention(
      (heads): ModuleList(
        (0-3): 4 x AttentionHead(
          (query): Linear(in_features=48, out_features=12, bias=True)
          (key): Linear(in_features=48, out_features=12, bias=True)
          (value): Linear(in_features=48, out_features=12, bias=True)
          (dropout): Dropout(p=0.0, inplace=False)
        )
      )
      (output_projection): Linear(in_features=48, out_features=48, bias=True)
      (output_dropout): Dropout(p=0.0, inplace=False)
    )
    (layernorm_1): LayerNorm((48,), eps=1e-05, elementwise_affine=True, bias=True)
    (mlp): MLP(
      (dense_1): Linear(in_features=48, out_features=192, bias=True)
      (activation): NewGELUActivation()
      (dense_2): Linear(in_features=192, out_features=48, bias=True)
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (layernorm_2): LayerNorm((48,), eps=1e-05, elementwise_affine=True, bias=True)
  )
)